# Revision Experiments — robustness analyses

**Paper:** Regime Detection and Forecasting of Financial Indicators in Electric Transmission Sector Companies Using Hidden Markov Models  
**Date:** 2026-05-11

This notebook implements three robustness experiments:

- **Proposal 1:** Non-HMM baseline comparison — rolling ARIMA(1,0,1) and Naïve Random Walk vs. existing GHMM NRMSE results.
- **Proposal 2:** GHMM robustness — re-run 3-state Gaussian HMM with training window sizes W ∈ {25, 30, 35, 40} (Gross Income indicator, all companies).
- **Proposal 4:** EM sensitivity — run GHMM with seeds 0–9 to assess initialization sensitivity (NRMSE ± std, log-likelihood per seed).

All experiments use the same **rolling one-step-ahead** evaluation scheme (train_size=35, 23 OOS steps) as Table 3 of the paper.  
Log-returns: `np.log(serie / serie.shift(1)).dropna()`

In [2]:
import pandas as pd
import numpy as np
from dotenv import load_dotenv
import os
from statsmodels.tsa.arima.model import ARIMA
from hmmlearn import hmm
import warnings

load_dotenv()
path_dados_tratados = os.getenv('PATH_DADOS_TRATADOS')

resultado_bruto = pd.read_csv(f"{path_dados_tratados}resultado_bruto.csv", index_col=0, parse_dates=True)
lucro_consolidado = pd.read_csv(f"{path_dados_tratados}lucro_consolidado.csv", index_col=0, parse_dates=True)
despesas_operacionais = pd.read_csv(f"{path_dados_tratados}despesas_operacionais.csv", index_col=0, parse_dates=True)

indicators = {
    'resultado_bruto': resultado_bruto,
    'lucro_consolidado': lucro_consolidado,
    'despesas_operacionais': despesas_operacionais,
}

companies = list(resultado_bruto.columns)
print('Companies:', companies)
print('resultado_bruto shape:', resultado_bruto.shape)

TypeError: deprecate_kwarg() missing 1 required positional argument: 'new_arg_name'

## Proposal 1: Non-HMM Baselines (ARIMA + Naïve RW)

In [ ]:
def rolling_arima_nrmse(log_returns: np.ndarray, train_size: int = 35) -> float:
    """Rolling one-step-ahead ARIMA(1,0,1) forecast, returns NRMSE."""
    preds = []
    actuals = []
    for t in range(train_size, len(log_returns)):
        train = log_returns[:t]
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model = ARIMA(train, order=(1, 0, 1)).fit()
        forecast = model.forecast(steps=1)[0]
        preds.append(forecast)
        actuals.append(log_returns[t])
    preds, actuals = np.array(preds), np.array(actuals)
    rmse = np.sqrt(np.mean((preds - actuals) ** 2))
    nrmse = rmse / (actuals.max() - actuals.min())
    return nrmse


def naive_rw_nrmse(log_returns: np.ndarray, train_size: int = 35) -> float:
    """Naïve random-walk baseline: forecast(t) = log_return(t-1). Returns NRMSE."""
    actuals = log_returns[train_size:]
    preds   = log_returns[train_size - 1: -1]
    rmse    = np.sqrt(np.mean((preds - actuals) ** 2))
    nrmse   = rmse / (actuals.max() - actuals.min())
    return nrmse

In [ ]:
rows = []
for ind_name, df in indicators.items():
    for company in companies:
        serie = df[company].dropna()
        log_ret = np.log(serie / serie.shift(1)).dropna().values
        arima_nrmse = rolling_arima_nrmse(log_ret, train_size=35)
        rw_nrmse    = naive_rw_nrmse(log_ret, train_size=35)
        rows.append({
            'indicator': ind_name,
            'company':   company,
            'ARIMA_NRMSE': round(arima_nrmse, 4),
            'NaiveRW_NRMSE': round(rw_nrmse, 4),
        })

df_proposal1 = pd.DataFrame(rows)
print('=== Proposal 1: Non-HMM Baseline NRMSE (rolling one-step-ahead, train_size=35) ===')
print(df_proposal1.to_string(index=False))

## Proposal 2: GHMM Robustness — Training Window Sizes

In [ ]:
def ghmm_rolling_nrmse(log_returns: np.ndarray, train_size: int,
                        n_states: int = 3, n_iter: int = 200,
                        random_seed: int = 42) -> float:
    """Rolling GHMM NRMSE for a given initial training window size."""
    preds, actuals = [], []
    for t in range(train_size, len(log_returns)):
        obs = log_returns[:t].reshape(-1, 1)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model = hmm.GaussianHMM(n_components=n_states, covariance_type="full",
                                    n_iter=n_iter, random_state=random_seed)
            model.fit(obs)
        last_state = model.predict(obs)[-1]
        next_state_probs = model.transmat_[last_state]
        forecast = np.dot(next_state_probs, model.means_.flatten())
        preds.append(forecast)
        actuals.append(log_returns[t])
    preds, actuals = np.array(preds), np.array(actuals)
    rmse = np.sqrt(np.mean((preds - actuals) ** 2))
    return rmse / (actuals.max() - actuals.min())

In [ ]:
window_sizes = [25, 30, 35, 40]
rows2 = []
for company in companies:
    serie = resultado_bruto[company].dropna()
    log_ret = np.log(serie / serie.shift(1)).dropna().values
    for w in window_sizes:
        nrmse = ghmm_rolling_nrmse(log_ret, train_size=w, n_states=3, random_seed=42)
        rows2.append({
            'company':    company,
            'train_size': w,
            'GHMM_NRMSE': round(nrmse, 4),
        })

df_proposal2 = pd.DataFrame(rows2)
print('=== Proposal 2: GHMM Robustness — resultado_bruto, 3-state GHMM ===')
pivot2 = df_proposal2.pivot(index='company', columns='train_size', values='GHMM_NRMSE')
pivot2.columns = [f'W={c}' for c in pivot2.columns]
print(pivot2.to_string())

## Proposal 4: GHMM EM Sensitivity — Initialization Seeds

In [ ]:
def ghmm_sensitivity(log_returns: np.ndarray, train_size: int = 35,
                     n_states: int = 3, seeds: list = list(range(10))) -> dict:
    """
    Run GHMM with multiple EM seeds. Returns per-seed NRMSE and log-likelihood.
    """
    results = {}
    for seed in seeds:
        preds, actuals = [], []
        converged_lls = []
        for t in range(train_size, len(log_returns)):
            obs = log_returns[:t].reshape(-1, 1)
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                model = hmm.GaussianHMM(n_components=n_states, covariance_type="full",
                                        n_iter=200, random_state=seed)
                model.fit(obs)
            converged_lls.append(model.monitor_.history[-1])
            last_state = model.predict(obs)[-1]
            next_state_probs = model.transmat_[last_state]
            forecast = np.dot(next_state_probs, model.means_.flatten())
            preds.append(forecast)
            actuals.append(log_returns[t])
        preds, actuals = np.array(preds), np.array(actuals)
        rmse = np.sqrt(np.mean((preds - actuals) ** 2))
        nrmse = rmse / (actuals.max() - actuals.min())
        results[seed] = {"nrmse": nrmse, "mean_ll": np.mean(converged_lls)}
    return results

In [ ]:
print('=== Proposal 4: GHMM EM Sensitivity — resultado_bruto, seeds 0–9 ===')
rows4 = []
for company in companies:
    serie = resultado_bruto[company].dropna()
    log_ret = np.log(serie / serie.shift(1)).dropna().values
    seed_results = ghmm_sensitivity(log_ret, train_size=35, n_states=3, seeds=list(range(10)))
    nrmse_vals = [v['nrmse'] for v in seed_results.values()]
    ll_vals    = [v['mean_ll'] for v in seed_results.values()]
    print(f"\nCompany {company}:")
    print(f"  NRMSE  — mean={np.mean(nrmse_vals):.4f}, std={np.std(nrmse_vals):.4f}, "
          f"min={np.min(nrmse_vals):.4f}, max={np.max(nrmse_vals):.4f}")
    print(f"  Mean LL — mean={np.mean(ll_vals):.2f}, std={np.std(ll_vals):.2f}")
    for seed, v in seed_results.items():
        rows4.append({'company': company, 'seed': seed,
                      'NRMSE': round(v['nrmse'], 4),
                      'mean_LL': round(v['mean_ll'], 2)})

df_proposal4 = pd.DataFrame(rows4)
print('\n--- Full per-seed table ---')
print(df_proposal4.to_string(index=False))